# Two workflows

1. Passing individual options
2. Passing all options with dictionary within helper functions

In [1]:
import os

from dotenv import load_dotenv
from pyspark.sql import SparkSession, DataFrame

load_dotenv();

## 1. Passing individual options workflow

In [2]:
spark = (
    SparkSession.builder
    .config("spark.sql.repl.eagerEval.enabled", "True")

    # Load jdbc driver jar from local path location
    # .config("spark.jars", str(Path(os.environ["PG_JDBC_DRIVER_PATH"])))

    # or download the jdbc driver directly from Maven repository
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.4")
    .appName("PySpark Postgres Workflow")
    .getOrCreate()
)

In [3]:
query = "select current_date"

#### Save query as PySpark dataframe

In [4]:
jdbcDF = (
    spark.read
    .format("jdbc")
    .option("driver", "org.postgresql.Driver")
    .option(
        "url",
        f"jdbc:postgresql://{os.environ['DB_HOST']}:{os.environ.get('DB_PORT', '5432')}/{os.environ['DB_NAME']}"
    )
    .option("user", os.environ["DB_USER"])
    .option("password", os.environ["DB_PASSWORD"])
    .option("query", query)
    .load()
)

In [5]:
jdbcDF.show()

+------------+
|current_date|
+------------+
|  2026-11-09|
+------------+



#### Write PySpark dataframe as Postgres table

In [6]:
table = "public.delete_me"

In [7]:
(jdbcDF
    .write
    .format("jdbc")
    .option("driver", "org.postgresql.Driver")
    .option(
        "url",
        f"jdbc:postgresql://{os.environ['DB_HOST']}:{os.environ.get('DB_PORT', '5432')}/{os.environ['DB_NAME']}"
    )
    .option("user", os.environ["DB_USER"])
    .option("password", os.environ["DB_PASSWORD"])
    .option("dbtable", table)
    # .option("truncate", "true")
    # .option("batchsize", 10000)
    # .option("numPartitions", 4)
    .mode("overwrite")
    .save()
)

## 2. Passing all options with dictionary to helper functions

In [8]:
JDBC_OPTIONS = {
    "url": (
        f"jdbc:postgresql://{os.environ['DB_HOST']}:"
        f"{os.environ.get('DB_PORT', '5432')}/{os.environ['DB_NAME']}"
    ),
    "user": os.environ["DB_USER"],
    "password": os.environ["DB_PASSWORD"],
    "driver": "org.postgresql.Driver",
}

In [9]:
def get_spark(app_name: str = "Postgres") -> SparkSession:
    return (
        SparkSession.builder
        .config("spark.sql.repl.eagerEval.enabled", "True")

        # Load jdbc driver jar from local path location
        # .config("spark.jars", str(Path(os.environ["PG_JDBC_DRIVER_PATH"])))

        # or download the jdbc driver directly from Maven repository
        .config("spark.jars.packages", "org.postgresql:postgresql:42.7.4")
        .appName(app_name)
        .getOrCreate()
    )

In [10]:
def read_query(spark: SparkSession, sql: str, **extra_options) -> DataFrame:
    """Run a SQL query against Postgres and return a Spark DataFrame."""
    return (
        spark.read.format("jdbc")
        .options(**JDBC_OPTIONS)
        .option("query", sql)
        .options(**extra_options)
        .load()
    )

In [11]:
def read_table(spark: SparkSession, table: str, **extra_options) -> DataFrame:
    """Read a whole table (Spark pushes down filters where it can)."""
    return (
        spark.read.format("jdbc")
        .options(**JDBC_OPTIONS)
        .option("dbtable", table)
        .options(**extra_options)
        .load()
    )

In [12]:
def write_dataframe(df: DataFrame, table: str, mode: str = "overwrite", **extra_options) -> None:
    """Write a PySpark DataFrame to a Postgres table."""
    (
        df
        .write
        .format("jdbc")
        .options(**JDBC_OPTIONS)
        .option("dbtable", table)
        .options(**extra_options)
        .mode(mode)
        .save()
    )

In [13]:
spark = get_spark()

In [14]:
query = "select current_date"

In [15]:
sdf = read_query(spark, query)

In [16]:
sdf.show()

+------------+
|current_date|
+------------+
|  2026-11-09|
+------------+



In [17]:
write_dataframe(sdf, "public.delete_me")